# Searching for Koopman eigenfunctions

We search for polynomial Koopman eigenfunctions of the van der Pol system. A polynomial observable $P$ is an eigenfunction with eigenvalue $\lambda$ when

$$\frac{dP}{dt} - \lambda P = 0.$$

The polynomial coefficients of this expression provide algebraic equations for the unknown coefficients of $P$ and $\lambda$.

In [ ]:
import sympy as sy
from IPython.display import Math, display

from symode.componentwise_expression_factory import (
    create_componentwise_expression,
    create_parametrized_polynomial,
)
from symode.dynamical_system import DynamicalSystem
from symode.root_finder import get_reduced_expression

## The dynamical system

In [ ]:
system = DynamicalSystem("lorenz")
Math(str(system))

## Polynomial ansatz

Set `max_degree` to choose the maximum total degree of the polynomial. For every exponent tuple, the coefficient is named using that tuple: for example, `a_0_0_0` multiplies the constant term and `a_1_0_0` multiplies the first system variable.

In [ ]:
max_degree = 2
polynomial, ansatz_coefficients = create_parametrized_polynomial(
    max_degree, system.get_variables()
)
polynomial

## Time derivative and eigenfunction remainder

The system computes the time derivative using its vector field. We then subtract $\lambda P$ and expand the result as a polynomial in the state variables.

In [ ]:
lambda_ = sy.Symbol("lambda")
remainder = sy.expand(system.get_operator_application(polynomial, lambda_))
remainder

## Map monomials to coefficients

Each monomial in the system variables is mapped to the coefficient that must vanish. Solving these coefficient equations is the next step in finding admissible polynomial eigenfunctions.

In [ ]:
monomial_coefficients = create_componentwise_expression(
    remainder, system.get_variables()
)
monomial_coefficients, preliminary_solution = get_reduced_expression(
    monomial_coefficients
)

polynomial = polynomial.subs(preliminary_solution)
ansatz_coefficients = [
    coefficient
    for coefficient in ansatz_coefficients
    if coefficient not in preliminary_solution
]
monomial_coefficients.get_components()

In [ ]:
coefficient_equations = list(monomial_coefficients.get_components().values())
groebner_basis = sy.groebner(
    coefficient_equations,
    *ansatz_coefficients,
    *system.get_parameters(),
    lambda_,
    order="grevlex",
)
groebner_basis

In [ ]:
solutions = sy.solve(
    groebner_basis, ansatz_coefficients + system.get_parameters() + [lambda_], dict=True
)
solutions = [
    solution for solution in solutions if sy.expand(polynomial.subs(solution)) != 0
]

In [ ]:
for solution in solutions:
    parameters = r",\quad ".join(
        (
            rf"{sy.latex(parameter)} = {sy.latex(solution[parameter])}"
            if parameter in solution
            else sy.latex(parameter)
        )
        for parameter in system.get_parameters()
    )
    eigenvalue = sy.latex(solution.get(lambda_, lambda_))
    eigenfunction = sy.latex(polynomial.subs(solution))
    display(
        Math(rf"{parameters}\qquad \lambda = {eigenvalue}\qquad P = {eigenfunction}")
    )